# DX702 Coding Quiz: Week 8

In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm # For linear regression
import statsmodels.formula.api as smf

from scipy.spatial.distance import mahalanobis
from sklearn.neighbors import NearestNeighbors

# from sklearn.linear_model import LinearRegression

In [8]:

df_8a = pd.read_csv('https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/main/homework_8.1.csv')
df_8a.drop(columns=['Unnamed: 0'], inplace = True)
print(df_8a.info())
df_8a.head(n = 10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X       1000 non-null   int64  
 1   Y       1000 non-null   float64
 2   Z       1000 non-null   float64
dtypes: float64(2), int64(1)
memory usage: 23.6 KB
None


,X,Y,Z
0,1,4.109218,1.764052
1,0,2.259504,0.400157
2,0,-0.647584,0.978738
3,0,2.106071,2.240893
4,1,3.583464,1.867558
5,1,1.657828,-0.977278
6,1,0.812524,0.950088
7,0,0.963567,-0.151357
8,0,-0.089470,-0.103219
9,1,0.865897,0.410599


### Question 1  
**10 Points**  

Using `homework_8.1.csv`, find the **Average treatment effect with inverse probability weighting.** Then, include your code and a written explanation of your work (mentioning any choices or strategies you made in writing the code) in your homework reflection.  

Here are some steps to follow:  

* Estimate the propensity scores using logistic regression. Fit the model so that the Z values predict ﻿X﻿.  
* Use the model to predict the propensity scores (e.g., using `predict_proba` if you are using sklearn).  
* Calculate inverse probability weights (﻿1 / P﻿ for ﻿X equals 1﻿, and ﻿1 / (1 - P)﻿ for ﻿X equals 0﻿).  
* Estimate the average treatment effect (the Y difference between ﻿X = 1﻿ and ﻿X = 0﻿, using the appropriate weights for each).  

Then, the ATE is closest to:  
- Option A: 2.0  
- Option B: 2.6  
- Option C: 2.3  
- Option D: 2.9

<font color='plum'>The **Estimated Average Treatment Effect (ATE)** using **Inverse Probability Weighting (IPW)** is **2.27**, which is closest to:

> **Option C: 2.3**

---

### 🔍 Explanation of approach

2. **Propensity Score Estimation**:
   - Fitted a logistic regression model to estimate the probability of receiving treatment (`X = 1`) based on `Z`.

3. **Inverse Probability Weights**:
   - For treated units: weight =  1 / {P(X = 1 | Z)} 
   - For control units: weight =  1 / {1 - P(X = 1 | Z)} 

4. **Weighted Outcome Averages**:
   - Computed the weighted average of `Y` for both treated and control groups.

5. **ATE Calculation**:
   - `ATE = Weighted average outcome (treated) − Weighted average outcome (control)`


In [9]:
# Estimate propensity scores using logistic regression model to estimate the probability of receiving treatment (`X = 1`) based on `Z`.
logit_model = smf.logit('X ~ Z', data = df_8a).fit(disp = 0)
df_8a['propensity_score'] = logit_model.predict(df_8a)
df_8a.head(n = 10)

,X,Y,Z,propensity_score
0,1,4.109218,1.764052,0.841702
1,0,2.259504,0.400157,0.585325
2,0,-0.647584,0.978738,0.712446
3,0,2.106071,2.240893,0.894224
4,1,3.583464,1.867558,0.854656
5,1,1.657828,-0.977278,0.269971
6,1,0.812524,0.950088,0.706705
7,0,0.963567,-0.151357,0.452238
8,0,-0.089470,-0.103219,0.463858
9,1,0.865897,0.410599,0.587787


In [10]:
# Calculate inverse probability weights
df_8a['ipw'] = np.where(
    df_8a['X'] == 1, 
    1 / df_8a['propensity_score'], 
    1 / (1 - df_8a['propensity_score'])
    )

# Separate treated and control groups
treated = df_8a[df_8a['X'] == 1]
control = df_8a[df_8a['X'] == 0]

In [13]:
# Estimate Average Treatment Effect
y1_weighted_mean = np.sum(treated['Y'] * treated['ipw']) / np.sum(treated['ipw'])
y0_weighted_mean = np.sum(control['Y'] * control['ipw']) / np.sum(control['ipw'])

ate_ipw          = y1_weighted_mean - y0_weighted_mean

print(f"ATE with IPW: {ate_ipw:.4f}")

ATE with IPW: 2.2680


### Question 2  
**10 Points**  

The propensity scores of the first three items in the dataframe are closest to:  
- Option A: [0.84, 0.58, 0.71]  
- Option B: [0.08, 0.65, 0.38]  
- Option C: [0.74, 0.39, 0.52]  
- Option D: [0.28, 0.34, 0.92]

In [12]:
# Get propensity scores for the first three items
propensity_scores_first_three = df_8a['propensity_score'].head(3).values

print(f"Propensity scores for the first three items: {propensity_scores_first_three}")

Propensity scores for the first three items: [0.84170175 0.58532468 0.71244576]


------------------------------
### Question 3  
**10 Points**  

Using `homework_8.2.csv`, match all treated items to the single nearest untreated item using the **Mahalanobis distance**. (Do this with replacement — the same untreated item can be used again.)  

* Use the `mahalanobis` function from `scipy.spatial.distance`  
* For the inverse covariance matrix, use all ﻿Z1﻿ values and all ﻿Z2﻿ values, make them into a ﻿2 x N﻿ matrix, find its ﻿2 x 2﻿ covariance, and invert.  

Then, the ATE is closest to:  
- Option A: 2.5  
- Option B: 3.1  
- Option C: 3.4  
- Option D: 2.8

In [62]:
df_8b = pd.read_csv('https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/main/homework_8.2.csv')
df_8b.drop(columns=['Unnamed: 0'], inplace = True)

print(df_8b.info())
df_8b.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X       1000 non-null   int64  
 1   Y       1000 non-null   float64
 2   Z1      1000 non-null   float64
 3   Z2      1000 non-null   float64
dtypes: float64(3), int64(1)
memory usage: 31.4 KB
None


,X,Y,Z1,Z2
0,1,4.652085,1.764052,2.320015
1,1,2.590221,0.400157,1.292631
2,1,3.898981,0.978738,0.556423
3,1,5.857179,2.240893,2.345607
4,1,3.647489,1.867558,2.095611


In [63]:
# X = treatment indicator, Y = outcome, Z1 & Z2 = covariates
treatment_col   = 'X'
outcome_col     = 'Y'
covariates      = ['Z1', 'Z2']

# 1. Separate treated and control groups
treated_df = df_8b[df_8b[treatment_col] == 1].copy()
control_df = df_8b[df_8b[treatment_col] == 0].copy()

# Z variables for matching
Z_treated   = treated_df[covariates]
Z_control   = control_df[covariates]
Z_all       = df_8b[covariates]

# 2. Calculate the inverse covariance matrix
cov_matrix       = np.cov(Z_all.values, rowvar=False)
inv_cov_matrix   = np.linalg.inv(cov_matrix)

In [64]:
# 3. Perform matching
matched_control_indices = []
max_dist                = -1
least_common_support_z  = None

for i, treated_row in Z_treated.iterrows():
    distances = [mahalanobis(treated_row, control_row, inv_cov_matrix) for _, control_row in Z_control.iterrows()]
    min_dist_index = np.argmin(distances)
    min_dist = distances[min_dist_index]
    
    matched_control_indices.append(Z_control.index[min_dist_index])
    
    if min_dist > max_dist:
        max_dist = min_dist
        least_common_support_z = treated_row.values

In [65]:
# 4. Estimate the ATE
y_treated = treated_df['Y'].values
y_matched_control = control_df.loc[matched_control_indices]['Y'].values
ate_matching = np.mean(y_treated - y_matched_control)

print(f"ATE from Mahalanobis matching: {ate_matching:4f}")


ATE from Mahalanobis matching: 3.437679


### Question 4  
**10 Points**  

Find the nearest Z1 and Z2 values of the treated item with the least common support (the farthest Mahalanobis distance from the untreated).  

- Option A: (2.3, 1.2)  
- Option B: (0.9, 1.4)  
- Option C: (0.2, -0.4)  
- Option D: (1.5, -1.3)

In [66]:
print(f"Z values of treated item with least common support: {least_common_support_z}")
print(f"Maximum Mahalanobis distance (least common support): {max_dist:.4f}")

Z values of treated item with least common support: [2.69622405 0.53815549]
Maximum Mahalanobis distance (least common support): 1.3830
